# Preprocessing


## Data Loading

Article presenting the dataset, with the acquisition methods and a data analysis: [Lim et al, 2018](https://pubmed.ncbi.nlm.nih.gov/30281467/)

Original Dataset (in EDF format) from: [STEW: Simultaneous Tast EEG Dataset](https://ieee-dataport.org/open-access/stew-simultaneous-task-eeg-workload-dataset)

Preprocessed dataset from: [monster-monash/STEW](https://huggingface.co/datasets/monster-monash/STEW)




In [47]:
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import skew, kurtosis


In [48]:
REPO_ID = "monster-monash/STEW"

def download_file(filename):
    return hf_hub_download(repo_id=REPO_ID,filename=filename,repo_type="dataset")

X_path = download_file("STEW_X.npy")

Y_path = download_file("STEW_y.csv")

SUBJECT_ID_path = download_file("STEW_subject_id.csv")

X = np.load(X_path)
Y = pd.read_csv(Y_path, header= None).squeeze().to_numpy()
SUBJECT_ID = pd.read_csv(SUBJECT_ID_path, header=None).squeeze().to_numpy()

## Data Exploration


In [49]:
# We verify the shapes of the three arrays

assert(X.ndim == 3)
assert(X.shape == (28512,14,256)) # Expected shape from the tabular dataset description
assert(len(X)==len(Y)==len(SUBJECT_ID))


## Data Cleaning

We remove all the data from subjects 5, 24, 42, as the dataset presentation article (Liu,2018) states that ratings were not obtained for this subjects, and without the information on the processed followed by the repository creators to create the labels from this subjects we remove them.


In [50]:
SUBJECTS_TO_REMOVE=[5,24,42]

mask =np.logical_not(np.isin(SUBJECT_ID,SUBJECTS_TO_REMOVE))
X=X[mask]
Y=Y[mask]
SUBJECT_ID=SUBJECT_ID[mask]

In [57]:
#We test that the removal was successful
for subject in SUBJECTS_TO_REMOVE:
    assert(X[SUBJECT_ID==subject].size==0)
    assert(Y[SUBJECT_ID==subject].size==0)
    assert(SUBJECT_ID[SUBJECT_ID==subject].size==0)

    assert(X[SUBJECT_ID==(subject-1)].size!=0)
    assert(Y[SUBJECT_ID==(subject-1)].size!=0)
    assert(SUBJECT_ID[SUBJECT_ID==(subject-1)].size!=0)


assert(X.shape == (28512-28512*3/48,14,256)) # Expected new shape, given that we removed 3/48th of the data
assert(len(X)==len(Y)==len(SUBJECT_ID))



## Feature engineering

We crete a .csv table with the chosen features


In [27]:
EEG_CHANNELS = ["AF3", "F7", "F3", "FC5", "T7", "P7", "O1", "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"]
SAMPLING_FREQUENCY = 128 

In [28]:
def extract_features(signal):

    # Time-domain features
    new_features = {
        "mean": np.mean(signal),
        "std" : np.std(signal),
        "var" : np.var(signal),
        "rms" : np.sqrt(np.mean(signal**2)),
        "skewness": skew(signal, bias=True),
        "kurtosis": kurtosis(signal, fisher=True, bias=True),
    }


    # Frequency-domain features
    freqs, psd = welch(
        signal,
        fs=SAMPLING_FREQUENCY,
        nperseg=len(signal)
    )

    frequency_resolution = freqs[1] - freqs[0] 


    FREQUENCY_BANDS = {
        "delta": (1, 4),
        "theta": (4, 8),
        "alpha": (8, 12),
        "beta": (12, 30),
        "gamma": (30, 50),
    }

    total_mask = (freqs >= 1) & (freqs < 45)
    total_power = np.sum(psd[total_mask]) * frequency_resolution

    for band, (low, high) in FREQUENCY_BANDS.items():

        mask = (freqs >= low) & (freqs < high)

        power = (
            np.sum(psd[mask]) * frequency_resolution
            if np.any(mask) else 0.0
        )

        new_features[f"{band}_power"] = power

        new_features[f"{band}_relative_power"] = (
            power / total_power
            if total_power > 1e-12 else 0.0
        )

    # Spectral entropy over the 1–45 Hz range
    psd_band = psd[total_mask]
    power_sum = psd_band.sum()

    if power_sum > 1e-12:
        probabilities = psd_band / power_sum
        probabilities = probabilities[probabilities > 0]

        new_features["spectral_entropy"] = -np.sum(
            probabilities * np.log2(probabilities)
        )
    else:
        new_features["spectral_entropy"] = 0.0

    return new_features

In [ ]:
records = []

for sample_idx in range(len(X)):

    eeg_sample = X[sample_idx]

    record = {
        "sample_id": sample_idx,
        "subject_id": SUBJECT_ID[sample_idx],
    }

    for channel_idx, channel_name in enumerate(EEG_CHANNELS):

        signal = eeg_sample[channel_idx, :]

        channel_features = extract_features(signal)

        for feature_name, value in channel_features.items():
            column_name = f"{channel_name}_{feature_name}"
            record[column_name] = value

    record["label"] = Y[sample_idx]
    records.append(record)


    if (sample_idx + 1) % 1000 == 0:
        print(f"Processed {sample_idx + 1}/{len(X)} examples")

We create a dataframe with the extracted outputs and write it to a .csv file

In [ ]:
OUTPUT_FILE = "STEW_features.csv"

extracted_features = pd.DataFrame(records)

print(extracted_features.head())
print(extracted_features.shape)

extracted_features.to_csv(OUTPUT_FILE,index=False)